In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv("delhi_metro_cleaned.csv")

df.head()

,TripID,Date,From_Station,To_Station,Distance_km,Fare,Cost_per_passenger,Passengers,Ticket_Type,Remarks
0,59771,2022-05-08,Inderlok,Kashmere Gate,12.94,77.99,18.27,13.0,Smart Card,maintenance
1,21363,2023-01-12,Model Town,Dilshad Garden,2.33,35.89,83.71,15.0,Tourist Card,maintenance
2,127325,2023-07-13,Kashmere Gate,Netaji Subhash Place,5.56,64.35,43.70,9.0,Single,off-peak
3,140510,2022-11-10,Chandni Chowk,Hauz Khas,4.02,144.13,14.98,27.0,Tourist Card,maintenance
4,144298,2022-11-06,Rajiv Chowk,Kalkaji Mandir,9.66,104.96,83.84,23.0,Single,off-peak


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   TripID              150000 non-null  int64  
 1   Date                150000 non-null  object 
 2   From_Station        150000 non-null  object 
 3   To_Station          150000 non-null  object 
 4   Distance_km         150000 non-null  float64
 5   Fare                150000 non-null  float64
 6   Cost_per_passenger  150000 non-null  float64
 7   Passengers          150000 non-null  float64
 8   Ticket_Type         150000 non-null  object 
 9   Remarks             150000 non-null  object 
dtypes: float64(4), int64(1), object(5)
memory usage: 11.4+ MB


In [4]:
df.isnull().sum()

TripID                0
Date                  0
From_Station          0
To_Station            0
Distance_km           0
Fare                  0
Cost_per_passenger    0
Passengers            0
Ticket_Type           0
Remarks               0
dtype: int64

In [5]:
df["Date"] = pd.to_datetime(df["Date"])

In [6]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Weekday"] = df["Date"].dt.day_name()

df.head()

,TripID,Date,From_Station,To_Station,Distance_km,Fare,Cost_per_passenger,Passengers,Ticket_Type,Remarks,Year,Month,Day,Weekday
0,59771,2022-05-08,Inderlok,Kashmere Gate,12.94,77.99,18.27,13.0,Smart Card,maintenance,2022,5,8,Sunday
1,21363,2023-01-12,Model Town,Dilshad Garden,2.33,35.89,83.71,15.0,Tourist Card,maintenance,2023,1,12,Thursday
2,127325,2023-07-13,Kashmere Gate,Netaji Subhash Place,5.56,64.35,43.70,9.0,Single,off-peak,2023,7,13,Thursday
3,140510,2022-11-10,Chandni Chowk,Hauz Khas,4.02,144.13,14.98,27.0,Tourist Card,maintenance,2022,11,10,Thursday
4,144298,2022-11-06,Rajiv Chowk,Kalkaji Mandir,9.66,104.96,83.84,23.0,Single,off-peak,2022,11,6,Sunday


In [7]:
np.random.seed(42)

df["Hour"] = np.random.randint(6,23,len(df))

In [8]:
df["Is_Peak_Hour"] = np.where(
    df["Hour"].isin([8,9,18,19]),
    1,
    0
)

df.head()

,TripID,Date,From_Station,To_Station,Distance_km,Fare,Cost_per_passenger,Passengers,Ticket_Type,Remarks,Year,Month,Day,Weekday,Hour,Is_Peak_Hour
0,59771,2022-05-08,Inderlok,Kashmere Gate,12.94,77.99,18.27,13.0,Smart Card,maintenance,2022,5,8,Sunday,12,0
1,21363,2023-01-12,Model Town,Dilshad Garden,2.33,35.89,83.71,15.0,Tourist Card,maintenance,2023,1,12,Thursday,20,0
2,127325,2023-07-13,Kashmere Gate,Netaji Subhash Place,5.56,64.35,43.70,9.0,Single,off-peak,2023,7,13,Thursday,16,0
3,140510,2022-11-10,Chandni Chowk,Hauz Khas,4.02,144.13,14.98,27.0,Tourist Card,maintenance,2022,11,10,Thursday,13,0
4,144298,2022-11-06,Rajiv Chowk,Kalkaji Mandir,9.66,104.96,83.84,23.0,Single,off-peak,2022,11,6,Sunday,12,0


In [9]:
np.random.seed(42)

df["Weather"] = np.random.choice(
    ["Sunny","Cloudy","Rainy"],
    len(df),
    p=[0.6,0.25,0.15]
)

In [10]:
np.random.seed(42)

df["Is_Holiday"] = np.random.choice(
    [0,1],
    len(df),
    p=[0.9,0.1]
)

In [11]:
df["From_Station"] = (
    df["From_Station"]
        .str.strip()
        .str.lower()
)

df["To_Station"] = (
    df["To_Station"]
        .str.strip()
        .str.lower()
)

In [12]:
station_map = {
    "rajiv chowk": 20,
    "kashmere gate": 18,
    "new delhi": 17,
    "central secretariat": 16,
    "mandi house": 15,
    "barakhamba road": 14,
    "chandni chowk": 14,
    "aiims": 13,
    "hauz khas": 12,
    "noida city centre": 11,
    "laxmi nagar": 10,
    "rajouri garden": 10,
    "kalkaji mandir": 9,
    "janakpuri west": 8,
    "netaji subhash place": 8,
    "pragati maidan": 7,
    "kirti nagar": 7,
    "punjabi bagh": 6,
    "model town": 6,
    "old delhi": 5,
    "jasola vihar": 5,
    "dilshad garden": 4,
    "inderlok": 4,
    "shivaji park": 3
}

In [13]:
station_effect = (
    df["From_Station"].map(station_map).fillna(2)
    +
    df["To_Station"].map(station_map).fillna(2)
)

station_effect.head()

0    22
1    10
2    26
3    26
4    29
dtype: int64

In [14]:
distance_effect = df["Distance_km"] * 1.5

distance_effect.head()

0    19.410
1     3.495
2     8.340
3     6.030
4    14.490
Name: Distance_km, dtype: float64

In [15]:
fare_effect = df["Fare"] * 0.4

fare_effect.head()

0    31.196
1    14.356
2    25.740
3    57.652
4    41.984
Name: Fare, dtype: float64

In [16]:
ticket_effect = df["Ticket_Type"].map({
    "Token": 0,
    "QR Ticket": 2,
    "Smart Card": 4
}).fillna(0)

ticket_effect.head()

0    4.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Ticket_Type, dtype: float64

In [17]:
df["Ticket_Type"].unique()

array(['Smart Card', 'Tourist Card', 'Single', 'Return'], dtype=object)

In [18]:
ticket_effect = df["Ticket_Type"].map({
    "Single": 0,
    "Return": 2,
    "Tourist Card": 3,
    "Smart Card": 6
}).fillna(0)

ticket_effect.head()

0    6
1    3
2    0
3    3
4    0
Name: Ticket_Type, dtype: int64

In [19]:
remark_effect = df["Remarks"].map({
    "Normal": 0,
    "Weekend": 5,
    "Festival": 15,
    "Event": 12
}).fillna(0)

remark_effect.head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Remarks, dtype: float64

In [20]:
weather_effect = df["Weather"].map({
    "Sunny": 0,
    "Cloudy": 3,
    "Rainy": 8
}).fillna(0)

In [21]:
holiday_effect = np.where(
    df["Is_Holiday"] == 1,
    -8,
    0
)

holiday_effect[:10]

array([ 0, -8,  0,  0,  0,  0,  0,  0,  0,  0])

In [22]:
np.random.seed(42)

df["Noise"] = np.random.normal(
    loc=0,
    scale=2,
    size=len(df)
)

df["Noise"].head()

0    0.993428
1   -0.276529
2    1.295377
3    3.046060
4   -0.468307
Name: Noise, dtype: float64

In [24]:
peak_hour_effect = np.where(
    df["Is_Peak_Hour"] == 1,
    15,
    0
)

peak_hour_effect[:10]

array([ 0,  0,  0,  0,  0,  0,  0, 15,  0, 15])

In [35]:
df["Passenger_Demand"] = (
    df["Passengers"] * np.random.uniform(0.8, 1.2, len(df))
    + station_effect
    + distance_effect
    + fare_effect
    + ticket_effect
    + remark_effect
    + peak_hour_effect
    + weather_effect
    + holiday_effect
    + df["Noise"] * 3
)

In [36]:
X = df.drop(columns=["Passenger_Demand"])

y = df["Passenger_Demand"]

In [67]:
X = df.drop(columns=[
    "Passenger_Demand",
    "Passengers",
    "Date",
    "TripID",
    "Year",
    "Day",
    "Noise"
])

In [68]:
categorical_features = [
    "From_Station",
    "To_Station",
    "Ticket_Type",
    "Remarks",
    "Weekday",
    "Weather"
]

In [69]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

X_encoded = preprocessor.fit_transform(X)

print(X_encoded.shape)

(150000, 75)


In [70]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data :", X_train.shape)
print("Testing Data  :", X_test.shape)

Training Data : (120000, 75)
Testing Data  : (30000, 75)


In [71]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [72]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[ 95.88297399 121.55585235  61.3993948  134.13881528 112.12554274
 117.36692038 124.42130854 106.24555282 119.53810561  96.00802326]


In [73]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

MAE : 6.21888160777014
RMSE: 7.786246538269393
R² Score: 0.9129798007701606


In [74]:
print(X.columns.tolist())

['From_Station', 'To_Station', 'Distance_km', 'Fare', 'Cost_per_passenger', 'Ticket_Type', 'Remarks', 'Month', 'Weekday', 'Hour', 'Is_Peak_Hour', 'Weather', 'Is_Holiday']


In [75]:
joblib.dump(model, "crowd_prediction_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!


In [76]:
sample = pd.DataFrame({
    "From_Station": ["rajiv chowk"],
    "To_Station": ["kashmere gate"],
    "Distance_km": [12],
    "Fare": [40],
    "Cost_per_passenger": [0.8],
    "Ticket_Type": ["Smart Card"],
    "Remarks": ["Normal"],
    "Year": [2025],
    "Month": [7],
    "Day": [15],
    "Weekday": ["Tuesday"],
    "Hour": [9],
    "Is_Peak_Hour": [1],
    "Weather": ["Rainy"],
    "Is_Holiday": [0],
    "Noise": [0]
})

sample_encoded = preprocessor.transform(sample)

prediction = model.predict(sample_encoded)

print("Predicted Passenger Demand:", prediction[0])

Predicted Passenger Demand: 121.25732544960601


In [77]:
df["Passenger_Demand"].describe()

count    150000.000000
mean         99.630813
std          26.367923
min          21.798257
25%          79.598481
50%          99.563020
75%         119.159239
max         214.972151
Name: Passenger_Demand, dtype: float64

In [78]:
print(df["Passenger_Demand"].min())
print(df["Passenger_Demand"].max())

21.798257074613538
214.97215106983268
